# 12 · Export to ONNX, check agreement, and benchmark fairly

**Level:** advanced · **Presentation:** 35–50 minutes · **Prerequisites:** notebook 04 and
the evaluation lesson. This notebook is independent of a custom-training checkpoint.

A deployment artifact is useful only if it loads, produces sensible predictions, and meets
a measured latency target. You will export established YOLO11n to ONNX, inspect its graph,
compare **postprocessed detections**, and measure warm-call CPU latency.

The export uses static batch 1, 320×320 input, FP32, and no graph simplification. Both paths use
the Ultralytics wrapper for the same preprocessing and NMS. This is an application-pipeline
comparison, **not a raw-runtime benchmark or a claim that ONNX is always faster**.

Install optional dependencies with the course environment activated:

```powershell
python -m pip install -r requirements-onnx.txt
```

Then restart the Jupyter kernel. The preflight reports missing packages and skips the optional
export coherently; it does not fabricate timing results. `COURSE_OFFLINE=1` also skips model work.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


In [ ]:
import os
import sys
import json
import shutil
import platform
import importlib.util
import importlib.metadata
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from course_utils import load_yolo, get_sample_image

OFFLINE = os.getenv("COURSE_OFFLINE", "0") == "1"
SMOKE = os.getenv("COURSE_SMOKE", "0") == "1"
IMG_SIZE = 320
DEVICE = "cpu"  # Both backends use CPU for this comparison.
WARMUP = 1 if SMOKE else 3
REPEATS = 2 if SMOKE else 15
EXPORT_DIR = ROOT / "models" / "exports"
OUT = ROOT / "outputs" / "12_onnx"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
required = ["torch", "ultralytics", "onnx", "onnxruntime", "onnxscript"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
READY = not OFFLINE and not missing
pt_model = onnx_model = image = pt_result = onnx_result = None
if OFFLINE:
    print("Offline mode: export and backend benchmarks skipped explicitly.")
elif missing:
    print("Missing:", ", ".join(missing))
    print(f'Install into this kernel: "{sys.executable}" -m pip install -r "{ROOT / "requirements-onnx.txt"}"')
    print("Then restart the kernel and Run All.")
else:
    print("Optional export dependencies found. Executing the real CPU export.")


## 1. Export a fixed-shape graph

We copy the checkpoint into `models/exports` before exporting because Ultralytics writes the
`.onnx` beside its checkpoint. No custom training is required. `simplify=False` avoids adding
a graph-simplifier dependency. `nms=False` keeps this YOLO11 graph's raw prediction head;
the Python inference wrapper performs the NMS afterward.

This notebook targets YOLO11's output contract. Other detector families, end-to-end heads,
and exports with embedded NMS can have different shapes and postprocessing requirements.


In [ ]:
if READY:
    import torch
    import onnx
    import onnxruntime as ort
    from ultralytics import YOLO

    loaded = load_yolo("yolo11n.pt")
    original_checkpoint = Path(loaded.ckpt_path)
    local_checkpoint = EXPORT_DIR / "yolo11n.pt"
    if original_checkpoint.resolve() != local_checkpoint.resolve():
        shutil.copy2(original_checkpoint, local_checkpoint)
    export_model = YOLO(str(local_checkpoint))
    onnx_path = Path(export_model.export(format="onnx", imgsz=IMG_SIZE, batch=1,
                                         dynamic=False, simplify=False, nms=False,
                                         device=DEVICE, opset=17))
    if not onnx_path.is_file():
        raise RuntimeError(f"Exporter did not produce the expected file: {onnx_path}")
    graph = onnx.load(str(onnx_path))
    onnx.checker.check_model(graph)
    print("Validated ONNX graph:", onnx_path)
    print(f"File size: {onnx_path.stat().st_size / 1024**2:.2f} MiB")


## 2. Inspect the model contract

The runtime input should be batch 1, three channels, height 320, width 320. Inspect the output
names and shapes instead of assuming an arbitrary detector uses this same layout. With YOLO11
detection and no embedded NMS, the head describes many candidate boxes and class scores;
it is not a ready-to-draw table of final detections.


In [ ]:
if READY:
    inspection_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    contract = []
    for kind, tensors in [("input", inspection_session.get_inputs()),
                          ("output", inspection_session.get_outputs())]:
        for tensor in tensors:
            contract.append({"kind": kind, "name": tensor.name,
                             "shape": str(tensor.shape), "type": tensor.type})
    display(pd.DataFrame(contract))
    assert inspection_session.get_inputs()[0].shape == [1, 3, IMG_SIZE, IMG_SIZE]
    print("Providers:", inspection_session.get_providers())
    del inspection_session  # Release its thread pool before the wrapper benchmark.


## 3. Make preprocessing and postprocessing match

`rect=False` forces the PyTorch path to use square padding, matching the static ONNX graph.
Without it, a rectangular source image can take a different letterbox path and produce
misleading differences. Both paths receive the same PIL RGB image and identical confidence,
IoU, and image-size settings. Ultralytics maps coordinates back to the original image.


In [ ]:
prediction_settings = dict(imgsz=IMG_SIZE, conf=0.25, iou=0.70, rect=False,
                           device=DEVICE, verbose=False)
if READY:
    image = Image.open(get_sample_image()).convert("RGB")
    pt_model = YOLO(str(local_checkpoint))  # Fresh instance after export.
    onnx_model = YOLO(str(onnx_path), task="detect")
    pt_result = pt_model.predict(image, **prediction_settings)[0]
    onnx_result = onnx_model.predict(image, **prediction_settings)[0]
    fig, axes = plt.subplots(1, 2, figsize=(12, 7))
    for ax, title, prediction in zip(axes, ["PyTorch", "ONNX Runtime"], [pt_result, onnx_result]):
        boxes = prediction.boxes
        show_boxes(np.asarray(image), boxes.xyxy.cpu().numpy(),
                   labels=[prediction.names[int(c)] for c in boxes.cls.cpu().tolist()],
                   scores=boxes.conf.cpu().tolist(), ax=ax, title=title)
    plt.show()


## 4. Match detections by class and IoU

Sorting rows is insufficient because backend floating-point differences can change score order.
Below, greedy one-to-one matching finds same-class boxes above IoU 0.5. We report unmatched
predictions and numerical deltas. This single-image check is a **deployment sanity check**,
not validation mAP; evaluate both backends on held-out labeled data before deployment.


In [ ]:
def box_iou_one(a, b):
    left_top = np.maximum(a[:2], b[:2])
    right_bottom = np.minimum(a[2:], b[2:])
    intersection = np.maximum(right_bottom - left_top, 0).prod()
    area_a = np.maximum(a[2:] - a[:2], 0).prod()
    area_b = np.maximum(b[2:] - b[:2], 0).prod()
    return float(intersection / max(area_a + area_b - intersection, 1e-9))

def compare_detections(reference, candidate, minimum_iou=0.5):
    a = reference.boxes.data.cpu().numpy()  # [x1, y1, x2, y2, confidence, class]
    b = candidate.boxes.data.cpu().numpy()
    pairs = sorted([(box_iou_one(x[:4], y[:4]), i, j)
                    for i, x in enumerate(a) for j, y in enumerate(b)
                    if int(x[5]) == int(y[5])], reverse=True)
    used_a, used_b, rows = set(), set(), []
    for overlap, i, j in pairs:
        if overlap < minimum_iou or i in used_a or j in used_b:
            continue
        used_a.add(i)
        used_b.add(j)
        rows.append({"class_name": reference.names[int(a[i, 5])], "iou": overlap,
                     "max_coordinate_delta_px": float(np.abs(a[i, :4] - b[j, :4]).max()),
                     "confidence_delta": float(abs(a[i, 4] - b[j, 4]))})
    summary = {"pytorch_boxes": len(a), "onnx_boxes": len(b), "matched": len(rows),
               "unmatched_pytorch": len(a) - len(used_a),
               "unmatched_onnx": len(b) - len(used_b)}
    return pd.DataFrame(rows, columns=["class_name", "iou", "max_coordinate_delta_px", "confidence_delta"]), summary

if READY:
    agreement, agreement_summary = compare_detections(pt_result, onnx_result)
    display(agreement.round(6))
    print(agreement_summary)
    agreement.to_csv(OUT / "prediction_agreement.csv", index=False)
    if agreement_summary["unmatched_pytorch"] or agreement_summary["unmatched_onnx"]:
        print("Investigate unmatched boxes, especially predictions near score/NMS thresholds.")
    elif agreement.empty:
        print("Both paths found no detections; this is inconclusive, so add positive examples.")
    else:
        print("All boxes matched on this image. Inspect coordinate and score deltas before accepting export.")


## 5. Measure warm CPU latency and its variability

We exclude model loading, export, image disk I/O, warmup, plotting, and JSON writing. Each timed
call includes wrapper preprocessing, inference, and postprocessing. Alternate backend order
across repetitions to reduce simple ordering bias. The CPU call is synchronous; a GPU benchmark
would require explicit synchronization and a separate protocol.

Median describes a typical call; p95 highlights slower calls. With 15 observations p95 is only
a rough estimate; the two-repeat smoke mode verifies execution, **not performance**. Derived FPS
assumes serial processing and excludes video decoding and display. Thread settings are reported,
not forced equal: the result compares these default backend configurations on this machine.


In [ ]:
if READY:
    backends = {"PyTorch": pt_model, "ONNX Runtime": onnx_model}
    for backend in backends.values():
        for _ in range(WARMUP):
            backend.predict(image, **prediction_settings)
    timing_rows = []
    names = list(backends)
    for repeat in range(REPEATS):
        for name in (names if repeat % 2 == 0 else names[::-1]):
            start = time.perf_counter()
            backends[name].predict(image, **prediction_settings)
            timing_rows.append({"backend": name, "repeat": repeat,
                                "milliseconds": 1000 * (time.perf_counter() - start)})
    timings = pd.DataFrame(timing_rows)
    summary_rows = []
    for name, group in timings.groupby("backend", sort=False):
        values = group["milliseconds"].to_numpy()
        summary_rows.append({"backend": name, "calls": len(values),
                             "median_ms": float(np.median(values)),
                             "p95_ms": float(np.percentile(values, 95)),
                             "serial_fps_from_median": float(1000 / np.median(values))})
    timing_summary = pd.DataFrame(summary_rows)
    display(timing_summary.round(2))
    fig, ax = plt.subplots(figsize=(8, 4))
    for name, group in timings.groupby("backend", sort=False):
        ax.plot(group["repeat"], group["milliseconds"], marker="o", label=name)
    ax.set(xlabel="Repetition", ylabel="Pipeline latency (ms)",
           title="Warm CPU calls — loading and plotting excluded")
    ax.legend()
    plt.show()
    timings.to_csv(OUT / "timing_samples.csv", index=False)
    timing_summary.to_csv(OUT / "timing_summary.csv", index=False)
    environment = {"python": sys.version, "platform": platform.platform(),
                   "processor": platform.processor(), "cpu_count": os.cpu_count(),
                   "device": DEVICE, "torch_num_threads": torch.get_num_threads(),
                   "onnx_providers_available": ort.get_available_providers(),
                   "versions": {name: importlib.metadata.version(name) for name in required},
                   "settings": prediction_settings, "warmup": WARMUP, "repeats": REPEATS,
                   "image_size_original": list(image.size), "agreement": agreement_summary}
    (OUT / "benchmark_environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
    print("Saved benchmark evidence to", OUT)
    if SMOKE:
        print("SMOKE RUN: insufficient repetitions for a performance conclusion.")
else:
    print("No real benchmark produced; complete the preflight and rerun for measured results.")


## 6. Exercises, interpretation, and a deployment checklist

1. Export again at 640. You must change `IMG_SIZE`, rerun export, and recreate **both** wrappers;
   a static 320 graph cannot accept arbitrary tensor sizes. Compare latency and held-out accuracy.
2. Increase `REPEATS` to 50, close other heavy apps, and rerun the timing section. Explain changes
   in median versus p95. Keep the same power mode and document the CPU.
3. Try a blank image and an image with many small objects. Why can NMS and detection counts
   change pipeline latency even when the network input tensor has fixed size?
4. Replace the checkpoint with your custom `best.pt`. Confirm names, coordinate mapping,
   preprocessing, and held-out metrics before measuring speed.

**Solution sketch:** 640 processes four times as many input pixels as 320, but whole-pipeline
latency need not scale by exactly four. Equivalent output requires matched transforms and
thresholds; fast output alone is not a successful export. Near-threshold boxes can differ across
floating-point implementations even when most predictions agree closely.

| Before shipping | Evidence to collect |
|---|---|
| Graph loads on the target machine | ONNX checker plus runtime inference |
| Preprocessing/postprocessing agree | Class names, coordinate mapping, matched boxes |
| Accuracy remains acceptable | Held-out labeled data, error review, and mAP |
| Latency meets the target | Representative inputs, warm median/p95, full app timing |
| Artifact is reproducible | Weights, versions, settings, and benchmark environment |

**Troubleshooting:** missing ONNX packages → install `requirements-onnx.txt` in the selected
kernel; Windows DLL errors → use matching 64-bit Python/package wheels and the required runtime;
static-shape errors → match the exported input size; large disagreement → verify `rect=False`,
RGB/BGR handling, class mapping, and NMS settings before relaxing tolerances.

Primary references: [Ultralytics export](https://docs.ultralytics.com/modes/export/),
[prediction](https://docs.ultralytics.com/modes/predict/),
[ONNX Runtime Python API](https://onnxruntime.ai/docs/api/python/api_summary.html).
